# BeatBridge: Batch anhören

Dieses Notebook lädt einen Batch über sein `manifest.json`, zeigt die gerenderten Varianten und spielt sie direkt im Notebook ab. Es setzt keine zusätzlichen Widgets voraus.

Starten Sie Jupyter im Projekt-Root (`BeatBridge`) und führen Sie die Zellen von oben nach unten aus. Für einen anderen Batch nur `BATCH_DIR` in der nächsten Zelle ändern.

In [ ]:
from pathlib import Path
import json
import sys

from IPython.display import Audio, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.ratings import ISSUE_TAGS, rating_summary, save_rating

# Für einen anderen Lauf diesen Ordner anpassen.
#output\batches\all_music_adjacent_equal_power_20260721\18__Frontliner_Tell_Me_Radio_Edit__to__Hard_Driver_Run_This_Town
BATCH_DIR = PROJECT_ROOT / "output" / "batches" / "all_music_adjacent_equal_power_20260721" / "18__Frontliner_Tell_Me_Radio_Edit__to__Hard_Driver_Run_This_Town"
MANIFEST_PATH = BATCH_DIR / "manifest.json"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Manifest nicht gefunden: {MANIFEST_PATH.resolve()}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
metrics_path = BATCH_DIR / "transition_metrics.json"
if metrics_path.is_file() and "rendered_transition_ranking" not in manifest:
    historical_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    metric_variants = {
        (item["candidate_id"], item["renderer"]): item
        for item in historical_metrics["variants"]
    }
    for item in manifest["rendered_variants"]:
        metric = metric_variants.get((item["candidate_id"], item["renderer"]))
        if metric is not None:
            item.update(
                transition_rank=metric.get("transition_rank"),
                transition_score=metric.get("transition_score"),
            )
variants = sorted(
    manifest["rendered_variants"],
    key=lambda item: (
        item.get("transition_rank") or item.get("candidate_rank") or 10**9,
        item["renderer"],
    ),
)
candidates = {item["candidate_id"]: item for item in manifest["candidate_generation"]["candidates"]}

def display_score(variant, key):
    score = variant.get(key)
    return "?" if score is None else f"{score:.3f}"

print(f"{len(candidates)} Kandidat(en), {len(variants)} gerenderte Variante(n)")
print(f"Renderer: {', '.join(manifest['renderers'])}")

In [ ]:
for index, variant in enumerate(variants):
    candidate = candidates[variant["candidate_id"]]
    print(
        f"[{index}] Render-Rang {variant.get('transition_rank', '?')} / "
        f"Render-Score {display_score(variant, 'transition_score')} / "
        f"Cutpoint-Rang {variant.get('candidate_rank', '?')} / "
        f"Cutpoint-Score {display_score(variant, 'candidate_score')} / "
        f"{variant['renderer']}: "
        f"{candidate['transition_length_bars']} Takte / "
        f"{candidate['transition_length_beats']} Beats — "
        f"{variant['output_file']}"
    )

In [ ]:
def play(index=0):
    """Spiele eine Variante über ihren Index aus der Liste oben ab."""
    variant = variants[index]
    candidate = candidates[variant["candidate_id"]]
    audio_path = BATCH_DIR / variant["output_file"]
    if not audio_path.is_file():
        raise FileNotFoundError(f"WAV-Datei nicht gefunden: {audio_path}")

    display(Markdown(
        f"### Variante {index}: Render-Rang {variant.get('transition_rank', '?')}, "
        f"Render-Score {display_score(variant, 'transition_score')}, "
        f"Cutpoint-Rang {variant.get('candidate_rank', '?')}, "
        f"Cutpoint-Score {display_score(variant, 'candidate_score')}, "
        f"`{variant['renderer']}`  \n"
        f"Kandidat `{variant['candidate_id']}`, "
        f"{candidate['transition_length_bars']} Takte / "
        f"{candidate['transition_length_beats']} Beats"
    ))
    return Audio(filename=str(audio_path))

play(0)

In [ ]:
def play_renderer(renderer):
    """Spiele die erste Variante eines Renderers ab, z. B. 'equal_power'."""
    for index, variant in enumerate(variants):
        if variant["renderer"] == renderer:
            return play(index)
    raise ValueError(f"Renderer nicht im Batch: {renderer}")

# Direkter A/B-Vergleich zum ersten Lauf:
play_renderer("equal_power")

## Variante bewerten

`rate(...)` speichert oder aktualisiert eine Bewertung von 1 (schlecht) bis 5 (sehr gut) atomar in `ratings.json` neben dem Manifest. Fehler-Tags: `rhythm`, `phrasing`, `harmony`, `energy`, `loudness`, `vocals`, `artifacts`, `other`.

In [ ]:
def rate(index, overall, issues=(), notes=""):
    """Bewerte eine angehörte Variante und speichere das Urteil dauerhaft."""
    variant = variants[index]
    saved = save_rating(
        BATCH_DIR,
        variant["candidate_id"],
        variant["renderer"],
        overall,
        issues=issues,
        notes=notes,
    )
    print(f"Gespeichert: Variante {index} = {saved['overall']}/5")
    return saved

def show_ratings():
    rows = rating_summary(BATCH_DIR)
    if not rows:
        print("Noch keine Bewertungen gespeichert.")
    for row in rows:
        print(
            f"{row['overall']}/5 | {row['renderer']} | {row['candidate_id']} | "
            f"{row['issues']} | {row['notes']}"
        )
    return rows

print("Fehler-Tags:", ", ".join(ISSUE_TAGS))
# Beispiel nach dem Anhören:
# rate(0, 2, issues=("phrasing", "vocals"), notes="Einstieg wirkt zu früh.")
show_ratings()

In [ ]:
play(1)

In [ ]:
rate(4, 1, issues=("energy"), notes="Songs 2 starte am Anfang. Kein gewollter Übergang.")

## Hörnotizen

Beim Vergleich besonders auf Beat-Stabilität, wahrgenommene Lautheit im Übergang, klangliche Kontinuität, Energieverlauf und mögliche Vocal-Clashes achten. Auffälligkeiten mit Variantenindex, Kandidaten-ID und Renderer in `docs/Bemerkungen.md` festhalten.